# Download Dataset


In [1]:
!pip install aicrowd-cli

In [2]:
API_KEY = 'cc0a3da7611cfc6098a7bd9db11b3ecf' # Please get your your API Key from [https://www.aicrowd.com/participants/me]
!aicrowd login --api-key $API_KEY

API Key valid
Saved API Key successfully!


In [3]:
# Downloading the Dataset
!mkdir data
!aicrowd dataset download --challenge emotion-detection -j 3 -o data

mkdir: cannot create directory ‘data’: File exists
val.csv:   0% 0.00/262k [00:00<?, ?B/s]

train.csv:   0% 0.00/2.30M [00:00<?, ?B/s]
val.csv: 100% 262k/262k [00:00<00:00, 1.50MB/s]

test.csv: 100% 642k/642k [00:00<00:00, 2.77MB/s]


train.csv: 100% 2.30M/2.30M [00:00<00:00, 7.18MB/s]


# Download & Import Libraries

In [4]:
!pip install emoji

In [5]:
import os
import re
import emoji
import time
import numpy as np
import pandas as pd
import nltk
import torch
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score
from torch import nn, optim, FloatTensor
from torch.utils.data import Dataset, DataLoader
from copy import deepcopy

In [6]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Read Dataset

In [7]:
train_dataset = pd.read_csv("data/train.csv")
validation_dataset = pd.read_csv("data/val.csv")[1:]
test_dataset = pd.read_csv("data/test.csv")

train_dataset.head(20)

,text,label
0,takes no time to copy/paste a press release,0
1,You're delusional,1
2,Jazz fan here. I completely feel. Lindsay Mann...,0
3,ah i was also confused but i think they mean f...,0
4,Thank you so much. ♥️ that means a lot.,0
5,And I’ll be there!!!,0
6,There are some amazingly cringey compilations ...,0
7,Check the frame (FPS) limit option in the adva...,0
8,you made me think I was in the dbd subreddit w...,0
9,It was in your op.,0


# Process text

In [8]:
stops = set(stopwords.words('english'))
porter = nltk.PorterStemmer()

def pre_process(str):
    def rm_html_tags(str):
        html_prog = re.compile(r'<[^>]+>',re.S)
        return html_prog.sub('', str)

    def rm_html_escape_characters(str):
        pattern_str = r'&quot;|&amp;|&lt;|&gt;|&nbsp;|&#34;|&#38;|&#60;|&#62;|&#160;|&#20284;|&#30524;|&#26684|&#43;|&#20540|&#23612;'
        escape_characters_prog = re.compile(pattern_str, re.S)
        return escape_characters_prog.sub('', str)

    def rm_at_user(str):
        return re.sub(r'@[a-zA-Z_0-9]*', '', str)

    def rm_url(str):
        return re.sub(r'http[s]?:[/+]?[a-zA-Z0-9_\.\/]*', '', str)

    def rm_repeat_chars(str):
        return re.sub(r'(.)(\1){2,}', r'\1\1', str)

    def rm_hashtag_symbol(str):
        return re.sub(r'#', '', str)

    def rm_time(str):
        return re.sub(r'[0-9][0-9]:[0-9][0-9]', '', str)

    def rm_punctuation(str):
        return re.sub(r'[^\w\s]', ' ', str)

    def split_emojis(str):
        text_part = ''.join(c for c in str if c not in emoji.UNICODE_EMOJI)
        emoji_part = ' '.join(c for c in str if c in emoji.UNICODE_EMOJI)
        return text_part + ' ' + emoji_part
    
    # do not change the preprocessing order only if you know what you're doing 
    str = str.lower()
    str = rm_url(str)        
    str = rm_at_user(str)        
    str = rm_repeat_chars(str) 
    str = rm_hashtag_symbol(str)       
    str = rm_time(str)
    str = rm_punctuation(str)
    # str = emoji.demojize(str, delimiters=(' emoji_', ' '))
    str = split_emojis(str)

    try:
        str = nltk.tokenize.word_tokenize(str)
        try:
            str = [porter.stem(t) for t in str]
        except:
            pass
    except:
        pass

    words = [w for w in str if w and w not in stops]
    return ' '.join(words)

In [9]:
text = "takes no time to copy/paste a press release"
pre_process(text)

'take time copi past press releas'

In [10]:
start_time = time.time()
train_dataset['processed_text'] = train_dataset['text'].apply(
    lambda x: pre_process(x)
)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

train_dataset.head(20)

Elapsed time: 14.1132 seconds


,text,label,processed_text
0,takes no time to copy/paste a press release,0,take time copi past press releas
1,You're delusional,1,delusion
2,Jazz fan here. I completely feel. Lindsay Mann...,0,jazz fan complet feel lindsay mann cousin ha v...
3,ah i was also confused but i think they mean f...,0,ah wa also confus think mean friend around age
4,Thank you so much. ♥️ that means a lot.,0,thank much mean lot
5,And I’ll be there!!!,0,
6,There are some amazingly cringey compilations ...,0,amazingli cringey compil terribl dialogu thi s...
7,Check the frame (FPS) limit option in the adva...,0,check frame fp limit option advanc graphic opt...
8,you made me think I was in the dbd subreddit w...,0,made think wa dbd subreddit statement idk whi
9,It was in your op.,0,wa op


In [11]:
start_time = time.time()
validation_dataset['processed_text'] = validation_dataset['text'].apply(
    lambda x: pre_process(x)
)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

validation_dataset.head(20)

Elapsed time: 1.6528 seconds


,text,label,processed_text
1,im still starving,1,im still starv
2,*Hey just noticed..* it's your **2nd Cakeday**...,0,hey notic 2nd cakeday slumbishop hug
3,They just did. Check out the sticky post.,0,check sticki post
4,"I hope so too, she deserves it.",0,hope deserv
5,is it dangerous to take a quick photo while st...,0,danger take quick photo stop red light
6,I’m in my second year. Still closeted. Still u...,1,second year still closet still uncomfort bodi ...
7,"Noted, I've been looking into that",0,note look
8,Thank you for saying that I just haven’t felt ...,0,thank say felt right sad constantli
9,"Screw the watch stuff, I wanna hear about the ...",1,screw watch stuff wan na hear porn
10,"Exactly....we need a car tunnel, and then let ...",0,exactli need car tunnel let old road allow tru...


# Create TF-IDF Features

In [12]:
print(train_dataset.shape)
print(validation_dataset.shape)
combined_dataset = pd.concat([train_dataset, validation_dataset])
print(combined_dataset.shape)

(31255, 3)
(3472, 3)
(34727, 3)


In [13]:
tfidf_vect = TfidfVectorizer(analyzer='word', token_pattern=r'\w{1,}', max_features=5000)
tfidf_vect.fit(combined_dataset['processed_text'])

TfidfVectorizer(analyzer='word', binary=False, decode_error='strict',
                dtype=<class 'numpy.float64'>, encoding='utf-8',
                input='content', lowercase=True, max_df=1.0, max_features=5000,
                min_df=1, ngram_range=(1, 1), norm='l2', preprocessor=None,
                smooth_idf=True, stop_words=None, strip_accents=None,
                sublinear_tf=False, token_pattern='\\w{1,}', tokenizer=None,
                use_idf=True, vocabulary=None)

In [14]:
start_time = time.time()
xtrain_tfidf =  tfidf_vect.transform(train_dataset['processed_text'])
xval_tfidf =  tfidf_vect.transform(validation_dataset['processed_text'])
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

print(xtrain_tfidf.shape)
print(xval_tfidf.shape)

Elapsed time: 0.3386 seconds
(31255, 5000)
(3472, 5000)


# Transform Data

In [15]:
class trainData(Dataset):
    def __init__(self, X_data, y_data):
        self.X_data = X_data
        self.y_data = y_data
        
    def __getitem__(self, index):
        return self.X_data[index], self.y_data[index]
        
    def __len__ (self):
        return len(self.X_data)

## test data    
class testData(Dataset):
    def __init__(self, X_data):
        self.X_data = X_data
        
    def __getitem__(self, index):
        return self.X_data[index]
        
    def __len__ (self):
        return len(self.X_data)

In [16]:
y_train = train_dataset['label'].values
y_val = validation_dataset['label'].values

train_data = trainData(
    FloatTensor(xtrain_tfidf.toarray()), 
    FloatTensor(y_train)
)
val_data = trainData(
    FloatTensor(xval_tfidf.toarray()), 
    FloatTensor(y_val)
)

In [17]:
## Initialize dataloader
LEARNING_RATE = 0.001
BATCH_SIZE = 128
NUM_EPOCHS = 100

train_loader = DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dataset=val_data, batch_size=BATCH_SIZE)

# Train Model

In [18]:
train_dataset['label'].value_counts()

0    24718
1     6537
Name: label, dtype: int64

In [19]:
# del model

In [20]:
# Hyperparameters for our network
input_size = xtrain_tfidf.shape[1]
# hidden_sizes = [64, 32]
hidden_sizes = [128, 64]
output_size = 1

# Build a feed-forward network
def declare_model():
    return nn.Sequential(
        nn.Linear(input_size, hidden_sizes[0]),
        nn.Dropout(0.5),
        nn.BatchNorm1d(hidden_sizes[0]),
        nn.ReLU(),
        nn.Linear(hidden_sizes[0], hidden_sizes[1]),
        nn.Dropout(0.5),
        nn.BatchNorm1d(hidden_sizes[1]),
        nn.ReLU(),
        nn.Linear(hidden_sizes[1], output_size),
        nn.Sigmoid()
    )

model = declare_model()
print(model)

Sequential(
  (0): Linear(in_features=5000, out_features=128, bias=True)
  (1): Dropout(p=0.5, inplace=False)
  (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): ReLU()
  (4): Linear(in_features=128, out_features=64, bias=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (7): ReLU()
  (8): Linear(in_features=64, out_features=1, bias=True)
  (9): Sigmoid()
)


In [21]:
# Declare class weight
criterion = nn.BCEWithLogitsLoss(pos_weight = torch.FloatTensor ([4.0]))
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [22]:
def eval(y_pred, y_true):
    y_pred_tag = torch.round(torch.sigmoid(y_pred))
    tp = (y_true * y_pred).sum().to(torch.float32)
    tn = ((1 - y_true) * (1 - y_pred)).sum().to(torch.float32)
    fp = ((1 - y_true) * y_pred).sum().to(torch.float32)
    fn = (y_true * (1 - y_pred)).sum().to(torch.float32)
    
    epsilon = 1e-7
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)
    f1 = 2 * (precision*recall) / (precision + recall + epsilon)
    acc = (tp + tn) / (tp + tn + fp + fn)
    
    return f1, acc

In [23]:
N_EPOCHS_STOP = 6
min_val_loss = np.Inf
epochs_no_improve = 0
best_model_state = {}

for e in range(1, NUM_EPOCHS+1):
    train_loss = val_loss = 0
    train_acc = val_acc = 0
    train_f1 = val_f1 = 0
    for X_batch, y_batch in train_loader:
        # X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()        
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch.unsqueeze(1))
        f1, acc = eval(y_pred, y_batch.unsqueeze(1))
        
        loss.backward() # Calculate gradients
        optimizer.step() # Update parameters
        
        train_loss += loss.item()
        train_acc += acc
        train_f1 += f1

    for X_batch, y_batch in val_loader:
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch.unsqueeze(1))
        f1, acc = eval(y_pred, y_batch.unsqueeze(1))
        val_loss += loss.item()
        val_acc += acc
        val_f1 += f1

    print(
        f'Epoch {e+0:03}: | \
        Training Loss: {train_loss/len(train_loader):.3f} | \
        F1 Score: {train_f1/len(train_loader):.3f} | \
        Acc: {train_acc/len(train_loader):.3f}  | \
        Validation Loss: {val_loss/len(val_loader):.3f} | \
        F1 Score: {val_f1/len(val_loader):.3f} | \
        Acc: {val_acc/len(val_loader):.3f}'
    )

    # Early stopping
    if val_loss/len(val_loader) < min_val_loss:
        # torch.save(model)  # Save the model
        epochs_no_improve = 0
        min_val_loss = val_loss/len(val_loader)
        best_model_state = deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1
    
    if epochs_no_improve >= N_EPOCHS_STOP:
        break

Epoch 001: |         Training Loss: 1.074 |         F1 Score: 0.391 |         Acc: 0.684  |         Validation Loss: 1.030 |         F1 Score: 0.465 |         Acc: 0.744
Epoch 002: |         Training Loss: 0.990 |         F1 Score: 0.567 |         Acc: 0.794  |         Validation Loss: 1.024 |         F1 Score: 0.486 |         Acc: 0.760
Epoch 003: |         Training Loss: 0.960 |         F1 Score: 0.636 |         Acc: 0.829  |         Validation Loss: 1.029 |         F1 Score: 0.484 |         Acc: 0.757
Epoch 004: |         Training Loss: 0.939 |         F1 Score: 0.681 |         Acc: 0.852  |         Validation Loss: 1.024 |         F1 Score: 0.491 |         Acc: 0.770
Epoch 005: |         Training Loss: 0.927 |         F1 Score: 0.709 |         Acc: 0.867  |         Validation Loss: 1.022 |         F1 Score: 0.501 |         Acc: 0.771
Epoch 006: |         Training Loss: 0.918 |         F1 Score: 0.731 |         Acc: 0.879  |         Validation Loss: 1.024 |         F1 Score: 0.498 |

In [24]:
best_model_state

OrderedDict([('0.weight',
              tensor([[ 0.0149,  0.0059, -0.0243,  ...,  0.0129, -0.0189,  0.0284],
                      [ 0.0065, -0.0209, -0.0450,  ...,  0.0029,  0.0047,  0.0005],
                      [-0.0230, -0.0171, -0.0388,  ..., -0.0125,  0.0213, -0.0226],
                      ...,
                      [ 0.0023, -0.0383, -0.0392,  ..., -0.0147,  0.0238,  0.0228],
                      [-0.0309,  0.0171,  0.0064,  ...,  0.0214,  0.0543, -0.0416],
                      [-0.0662, -0.0315, -0.0684,  ..., -0.0039, -0.0357,  0.0088]])),
             ('0.bias',
              tensor([-0.0261, -0.0374,  0.0388, -0.0063, -0.0312, -0.0185, -0.0118,  0.0365,
                      -0.0258, -0.0257, -0.0346, -0.0231,  0.0435, -0.0182, -0.0325, -0.0131,
                      -0.0341, -0.0300,  0.0097, -0.0444, -0.0340, -0.0174, -0.0205, -0.0156,
                      -0.0128, -0.0057, -0.0288, -0.0147, -0.0373,  0.0202, -0.0209,  0.0169,
                       0.0627, -0.0421, 

# Evaluate

## Training data

In [25]:
train_data = testData(
    FloatTensor(xtrain_tfidf.toarray())
)
train_loader = DataLoader(dataset=train_data, batch_size=1)

In [26]:
y_pred_list = []
# model = declare_model()
# model.load_state_dict(best_model_state)
model.eval()
with torch.no_grad():
    for X_batch in train_loader:
        # X_batch = X_batch.to(device)
        y_train_pred = model(X_batch)
        y_pred_tag = torch.round(y_train_pred)
        y_pred_list.append(y_pred_tag.cpu().numpy())

y_pred_list = [a.squeeze().tolist() for a in y_pred_list]
train_dataset['pred'] = y_pred_list

f1 = f1_score(train_dataset['label'], train_dataset['pred'])
accuracy = accuracy_score(train_dataset['label'], train_dataset['pred'])

print(f"Training F1 Score  : {round(f1, 4)} and Accuracy Score {round(accuracy, 4)}")

Training F1 Score  : 0.8252 and Accuracy Score 0.9188


## Validation data

In [27]:
val_data = testData(
    FloatTensor(xval_tfidf.toarray())
)
val_loader = DataLoader(dataset=val_data, batch_size=1)

In [28]:
y_pred_list = []
model.eval()
with torch.no_grad():
    for X_batch in val_loader:
        # X_batch = X_batch.to(device)
        y_val_pred = model(X_batch)
        y_pred_tag = torch.round(y_val_pred)
        y_pred_list.append(y_pred_tag.cpu().numpy())

y_pred_list = [a.squeeze().tolist() for a in y_pred_list]
validation_dataset['pred'] = y_pred_list

f1 = f1_score(validation_dataset['label'], validation_dataset['pred'])
accuracy = accuracy_score(validation_dataset['label'], validation_dataset['pred'])

print(f"Validation F1 Score  : {round(f1, 4)} and Accuracy Score {round(accuracy, 4)}")

Validation F1 Score  : 0.5154 and Accuracy Score 0.7509


# Submit Results

In [29]:
test_dataset['processed_text'] = test_dataset['text'].apply(
    lambda x: pre_process(x)
)
xtest_tfidf =  tfidf_vect.transform(test_dataset['processed_text'])
test_data = testData(
    FloatTensor(xtest_tfidf.toarray())
)
print(xtest_tfidf.shape)

test_loader = DataLoader(dataset=test_data, batch_size=1)

(8682, 5000)


In [30]:
y_pred_list = []
model.eval()
with torch.no_grad():
    for X_batch in test_loader:
        # X_batch = X_batch.to(device)
        y_test_pred = model(X_batch)
        y_pred_tag = torch.round(y_test_pred)
        y_pred_list.append(y_pred_tag.cpu().numpy())

y_pred_list = [a.squeeze().tolist() for a in y_pred_list]
test_dataset['label'] = y_pred_list
test_dataset.head()

,text,label,processed_text
0,I was already over the edge with Cassie Zamora...,1.0,wa alreadi edg cassi zamora show disdain two t...
1,I think you're right. She has oodles of cash a...,0.0,think right ha oodl cash young grandchildren e...
2,Haha I love this. I used to give mine phone bo...,0.0,haha love thi use give mine phone book room wo...
3,Probably out of desperation as they going no a...,0.0,probabl desper go answer made god
4,Sorry !! You’re real good at that!!,1.0,sorri real good


In [31]:
!mkdir assets

# Saving the sample submission in assets directory
if 'processed_text' in test_dataset.columns:
    test_dataset.drop(columns=['processed_text'], inplace=True)

test_dataset.to_csv(os.path.join("assets", "submission.csv"), index=False)

mkdir: cannot create directory ‘assets’: File exists


In [32]:
!aicrowd notebook submit -c emotion-detection -a assets --no-verify

Using notebook: /content/drive/MyDrive/Colab Notebooks/tfidf-ann-classifier.ipynb for submission...
Removing existing files from submission directory...
Scrubbing API keys from the notebook...
submission.zip ━━━━━━━━━━━━━━━━━━ 100.0% • 323.8/322.2 KB • 843.1 kB/s • 0:00:00
                                                  ╭─────────────────────────╮                                                  
                                                  │ Successfully submitted! │                                                  
                                                  ╰─────────────────────────╯                                                  
                                                        Important links                                                        
┌──────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│  This submission │ https://www.aicrowd.com/challenges/ai-blitz-9/problems/emotion-de

# References

#### Tutorial - Early Stopping - Vanilla RNN - PyTorch
https://www.kaggle.com/akhileshrai/tutorial-early-stopping-vanilla-rnn-pytorch

#### PyTorch [Tabular] — Binary Classification
https://towardsdatascience.com/pytorch-tabular-binary-classification-a0368da5bb89